# Prognozowanie temperatur lipca w Lublinie siecią CNN (1D)
Notebook warsztatowy: przygotowanie danych, sliding windows, CNN 1D, trening, ewaluacja i wizualizacje.

**Dane wejściowe (CSV, co 2 godziny):**
- `lublin_temperatures_july_2015_2024_2h.csv` (train+val)
- `lublin_temperatures_july_2025_2h.csv` (test)

Pliki w tej rozmowie zostały wygenerowane jako realistyczna symulacja (z lukami jak w czujniku). W realnym projekcie możesz je podmienić na dane historyczne (np. reanalysis) z API i zachować identyczny format.


## 1. Importy i ustawienia
Ustawiamy stałe: okno wejściowe (`LOOKBACK`) i horyzont prognozy (`HORIZON`).
Ponieważ pomiary są co 2h, w dobie mamy 12 próbek.

In [ ]:

# Jeśli uruchamiasz pierwszy raz:
# !pip install pandas numpy matplotlib tensorflow scikit-learn

from __future__ import annotations

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_CSV = "lublin_temperatures_july_2015_2024_2h.csv"
TEST_CSV  = "lublin_temperatures_july_2025_2h.csv"

TIMESTAMP_COL = "timestamp_local"
TARGET_COL = "temperature_C"

# Co 2h => 12 próbek na dobę
LOOKBACK_DAYS = 6
LOOKBACK = LOOKBACK_DAYS * 12  # 72 kroki (~6 dni)

HORIZON_HOURS = 24            # zmień na 48 jeśli chcesz
HORIZON = HORIZON_HOURS // 2  # 24h => 12 kroków

BATCH_SIZE = 64
EPOCHS = 100

KERNEL_SIZE = 5
DROPOUT = 0.10
LR = 1e-3


## 2. Wczytanie danych i szybka kontrola
Sprawdzamy podstawowe informacje: zakres dat, braki i prosty wykres temperatur.

In [ ]:

def load_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Nie znaleziono pliku: {path}")
    df = pd.read_csv(path)
    if TIMESTAMP_COL not in df.columns or TARGET_COL not in df.columns:
        raise ValueError(f"CSV musi mieć kolumny: {TIMESTAMP_COL}, {TARGET_COL}. Ma: {list(df.columns)}")
    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL])
    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
    return df

train_all = load_csv(TRAIN_CSV)
test_df = load_csv(TEST_CSV)

train_all.head(), test_df.head()


In [ ]:

print("Train+Val:", train_all.shape)
print("Test:", test_df.shape)

print("\nZakres dat train+val:", train_all[TIMESTAMP_COL].min(), "->", train_all[TIMESTAMP_COL].max())
print("Zakres dat test:", test_df[TIMESTAMP_COL].min(), "->", test_df[TIMESTAMP_COL].max())

print("\nBraki (NaN) w temperature_C:")
print("Train+Val:", train_all[TARGET_COL].isna().sum())
print("Test:", test_df[TARGET_COL].isna().sum())


In [ ]:

plt.figure(figsize=(10,4))
plt.plot(train_all[TIMESTAMP_COL].iloc[:12*7], train_all[TARGET_COL].iloc[:12*7])  # 7 dni podglądu
plt.title("Podgląd: pierwsze 7 dni (train+val)")
plt.xlabel("Czas")
plt.ylabel("Temperatura (°C)")
plt.grid(True)
plt.show()


## 3. Czyszczenie braków
W prawdziwych danych pogodowych zdarzają się luki. Uzupełniamy je interpolacją po czasie.
To prosta i rozsądna procedura dla gęstych szeregów (co 2h).

In [ ]:

def interpolate_temperature(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out[TARGET_COL] = (
        out.set_index(TIMESTAMP_COL)[TARGET_COL]
           .interpolate(method="time")
           .ffill()
           .bfill()
           .values
    )
    if out[TARGET_COL].isna().any():
        raise ValueError("Po interpolacji nadal są NaN.")
    return out

train_all = interpolate_temperature(train_all)
test_df = interpolate_temperature(test_df)

print("Braki po czyszczeniu:")
print("Train+Val:", train_all[TARGET_COL].isna().sum())
print("Test:", test_df[TARGET_COL].isna().sum())


## 4. Podział: train (2015–2023), val (2024), test (2025)
Walidacja na osobnym roku (2024) pomaga ocenić generalizację w obrębie lipca.

In [ ]:

years = train_all[TIMESTAMP_COL].dt.year

train_df = train_all[(years >= 2015) & (years <= 2023)].copy()
val_df   = train_all[years == 2024].copy()

assert len(train_df) > 0 and len(val_df) > 0, "Sprawdź zakres lat w danych."

train_series = train_df[TARGET_COL].to_numpy(dtype=np.float32)
val_series   = val_df[TARGET_COL].to_numpy(dtype=np.float32)
test_series  = test_df[TARGET_COL].to_numpy(dtype=np.float32)

len(train_series), len(val_series), len(test_series)


## 5. Normalizacja (tylko na train)
Normalizujemy temperaturę z-score:
\[ x' = (x - \mu)/\sigma \]
Kluczowe: **\(\mu\)** i **\(\sigma\)** liczymy tylko na train, żeby nie było przecieku informacji z val/test.

In [ ]:

class Normalizer:
    def __init__(self, mean_: float, std_: float):
        self.mean_ = float(mean_)
        self.std_ = float(std_)
    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean_) / (self.std_ + 1e-8)
    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * (self.std_ + 1e-8) + self.mean_

mean_ = float(np.mean(train_series))
std_  = float(np.std(train_series))
norm = Normalizer(mean_, std_)

train_norm = norm.transform(train_series)
val_norm   = norm.transform(val_series)
test_norm  = norm.transform(test_series)

mean_, std_


## 6. Sliding windows: szereg → próbki uczące
Budujemy próbki metodą okien przesuwanych:
- **X**: ostatnie `LOOKBACK` kroków (np. 6 dni)
- **y**: kolejne `HORIZON` kroków (np. 24h)

Model uczy się regresji wielowyjściowej: przewiduje cały wektor przyszłości.

In [ ]:

def make_windows(series: np.ndarray, lookback: int, horizon: int):
    T = len(series)
    N = T - lookback - horizon + 1
    if N <= 0:
        raise ValueError(f"Za mało danych na okna: T={T}, lookback={lookback}, horizon={horizon}")
    X = np.zeros((N, lookback, 1), dtype=np.float32)
    y = np.zeros((N, horizon), dtype=np.float32)
    for i in range(N):
        X[i, :, 0] = series[i:i+lookback]
        y[i, :] = series[i+lookback:i+lookback+horizon]
    return X, y

X_train, y_train = make_windows(train_norm, LOOKBACK, HORIZON)
X_val, y_val     = make_windows(val_norm,   LOOKBACK, HORIZON)
X_test, y_test   = make_windows(test_norm,  LOOKBACK, HORIZON)

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape


## 7. CNN 1D dla szeregu czasowego
CNN 1D traktuje temperaturę jako sygnał. Filtry (kernels) wykrywają **lokalne wzorce**:
- dobowość (cykl dzień/noc)
- krótkie zmiany (fronty)
- kilkudniowe fale upału/chłodu

Używamy padding `causal`, żeby filtr nie „podglądał przyszłości” w obrębie okna.

In [ ]:

def build_cnn_1d(lookback: int, horizon: int, lr: float = 1e-3) -> keras.Model:
    inputs = keras.Input(shape=(lookback, 1), name="temp_series")
    x = layers.Conv1D(32, kernel_size=KERNEL_SIZE, padding="causal", activation="relu")(inputs)
    x = layers.Conv1D(64, kernel_size=KERNEL_SIZE, padding="causal", activation="relu")(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(horizon, activation="linear", name="forecast")(x)

    model = keras.Model(inputs, outputs, name="cnn1d_temperature_forecast")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.Huber(delta=1.0),
        metrics=[
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model

model = build_cnn_1d(LOOKBACK, HORIZON, lr=LR)
model.summary()


## 8. Trening (EarlyStopping + ReduceLROnPlateau)
- `EarlyStopping`: przerywa trening, gdy walidacja nie poprawia się przez kilka epok.
- `ReduceLROnPlateau`: obniża learning rate, gdy val_loss „staje”.

In [ ]:

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


In [ ]:

plt.figure(figsize=(9,4))
plt.plot(history.history["loss"], label="loss (train)")
plt.plot(history.history["val_loss"], label="loss (val)")
plt.xlabel("Epoka")
plt.ylabel("Loss (Huber)")
plt.title("Krzywe uczenia")
plt.grid(True)
plt.legend()
plt.show()


## 9. Ewaluacja na teście (lipiec 2025)
Liczymy MAE/RMSE w °C (po odwróceniu normalizacji) oraz błąd zależny od horyzontu.

In [ ]:

# Predykcje w skali znormalizowanej
y_pred = model.predict(X_test, verbose=0)

# Odwrócenie normalizacji do °C
y_test_C = norm.inverse_transform(y_test)
y_pred_C = norm.inverse_transform(y_pred)

mae_C = float(np.mean(np.abs(y_test_C - y_pred_C)))
rmse_C = float(np.sqrt(np.mean((y_test_C - y_pred_C)**2)))

print(f"TEST MAE  (°C): {mae_C:.3f}")
print(f"TEST RMSE (°C): {rmse_C:.3f}")


In [ ]:

def mae_by_step(y_true: np.ndarray, y_hat: np.ndarray) -> np.ndarray:
    return np.mean(np.abs(y_true - y_hat), axis=0)

step_mae = mae_by_step(y_test_C, y_pred_C)

plt.figure(figsize=(9,4))
plt.plot(np.arange(1, HORIZON+1)*2, step_mae)  # godziny: 2,4,...,H*2
plt.xlabel("Horyzont prognozy (godziny)")
plt.ylabel("MAE (°C)")
plt.title("MAE w funkcji kroku prognozy")
plt.grid(True)
plt.show()

step_mae[:5], step_mae[-5:]


## 10. Wizualizacja: jedna przykładowa prognoza 24h/48h
Wybieramy jedno okno testowe i porównujemy prawdę z predykcją.

In [ ]:

idx = min(100, len(X_test) - 1)

true_seq = y_test_C[idx]
pred_seq = y_pred_C[idx]

x_hours = np.arange(1, HORIZON+1) * 2

plt.figure(figsize=(10,4))
plt.plot(x_hours, true_seq, label="prawda")
plt.plot(x_hours, pred_seq, label="predykcja")
plt.xlabel("Horyzont (godziny)")
plt.ylabel("Temperatura (°C)")
plt.title(f"Przykładowa prognoza {HORIZON_HOURS}h (1 okno testowe)")
plt.grid(True)
plt.legend()
plt.show()


## 11. Zapis prognoz do CSV
Zapisujemy prognozy dla wszystkich okien testowych: czas startu prognozy + pred/true dla każdego kroku.

In [ ]:

test_times = pd.to_datetime(test_df[TIMESTAMP_COL]).reset_index(drop=True)

N = y_pred_C.shape[0]
start_times = test_times.iloc[np.arange(N) + LOOKBACK].to_numpy()

out = pd.DataFrame({"forecast_start_local": start_times})
for k in range(HORIZON):
    out[f"t_plus_{(k+1)*2}h_pred_C"] = y_pred_C[:, k]
    out[f"t_plus_{(k+1)*2}h_true_C"] = y_test_C[:, k]

OUT_PATH = "lublin_cnn_forecasts_july_2025.csv"
out.to_csv(OUT_PATH, index=False)
OUT_PATH, out.head()


## 12. Ćwiczenia dodatkowe (dla ambitnych)
1. Zmień `HORIZON_HOURS` na 48 i porównaj MAE/RMSE.
2. Zmień `LOOKBACK_DAYS` na 10–14 i sprawdź wpływ na jakość.
3. Dodaj cechy czasowe jako dodatkowe kanały wejścia (sin/cos godziny i dnia).
4. Porównaj CNN z prostym baseline: „jutro jak dziś” (persistence) oraz średnią dobową.
